# Домашнее задание 3. Акустические сигналы и модели — Python


Данные: `data/Bearing.mat`. 

Заданные «проценты» в задаче 4 — mixing coefficients (коэффициенты смешивания),
не реальные измерения износа.

## Подготовка

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

DATA = Path("data")


def check(task, value):
    try:
        if task == 1:
            ok = (isinstance(value, dict) and all(k in value for k in
                  ["normal", "roller", "inner", "outer"])
                  and value["inner"].size == 120_000
                  and round(np.abs(value["inner"]).sum()) == 25057)
        elif task == 2:
            ok = (len(value) == 4 and all(np.asarray(v).shape == (120_000,)
                  for v in value) and round(sum(np.abs(v).sum() for v in value)) == 94201)
        elif task == 3:
            ok = (len(value) == 4 and all(np.asarray(v).shape == (117, 4000)
                  for v in value) and np.array_equal(value[0][0, 1000:], value[0][1, :3000]))
        elif task == 4:
            X, y = value
            ok = (np.asarray(X).shape == (37, 1) and np.asarray(y).shape == (37, 3)
                  and np.allclose(y[0], 0) and np.isclose(y.sum(), 1950)
                  and np.isclose(X[0, 0], 0.0726744486205))
        elif task == 5:
            arr = np.asarray(value, dtype=float)
            ok = arr.shape == (14,) and np.isfinite(arr).all() and round(arr.sum()) == 45545
        elif task == 6:
            arr = np.asarray(value, dtype=float)
            ok = (arr.shape == (37, 1) and np.allclose(arr.mean(axis=0), 0, atol=1e-12)
                  and np.allclose(arr.std(axis=0, ddof=1), 1, atol=1e-12))
        elif task == 7:
            ok = (len(value) == 3 and all(isinstance(v, RandomForestRegressor) for v in value)
                  and [v.n_estimators for v in value] == [400, 400, 600]
                  and [v.max_depth for v in value] == [35, 35, 50])
        else:
            raise ValueError("неизвестный номер")
    except Exception as exc:
        print(f"❌ Задание {task}: {type(exc).__name__}: {exc}")
        return
    print(f"{'✅' if ok else '❌'} Задание {task}" +
          (" выполнено" if ok else ": проверьте результат"))


print("Папка с данными найдена:", DATA.exists())
print("Файлы:", [p.name for p in DATA.iterdir()] if DATA.exists() else [])

Папка с данными найдена: True
Файлы: ['Bearing.mat']


## Задание 1. Загрузить MAT-файл

Сохраните словарь из `loadmat(DATA / "Bearing.mat")` в `bearing`.
Ожидаются ключи `normal`, `roller`, `inner`, `outer`.

In [2]:
# ✍ Задание 1
bearing = loadmat(DATA/"Bearing.mat")
print([key for key in bearing if not key.startswith("__")] if bearing is not Ellipsis else "Заполните код")

['normal', 'inner', 'roller', 'outer']


In [3]:
check(1, bearing)

✅ Задание 1 выполнено


## Задание 2. Векторы сигналов

Извлеките четыре сигнала в `normal`, `roller`, `inner`, `outer`.
Метод `.ravel()` превращает матрицу `1 × 120000` в одномерный вектор.

In [4]:
# ✍ Задание 2
normal = bearing['normal'].ravel()
roller = bearing['roller'].ravel()
inner = bearing['inner'].ravel()
outer = bearing['outer'].ravel()
print("Длины:", [len(v) for v in [normal, roller, inner, outer]] if normal is not Ellipsis else "Заполните код")

Длины: [120000, 120000, 120000, 120000]


In [5]:
check(2, (normal, roller, inner, outer))

✅ Задание 2 выполнено


## Задание 3. Окна с перекрытием

Напишите `segment_signal(signal, length=4000, overlap=0.75)`.
Шаг: `int(length * (1 - overlap))`; начала окон: `range(0, len(signal)-length+1, step)`.
Верните массив формы `(число_окон, length)` с помощью `np.stack`.
Получите `segments_normal`, `segments_roller`, `segments_inner`, `segments_outer`.
Какой участок соседних окон общий?

![Почему окна одного сигнала сильно похожи](visualizations/02_утечка_и_проверка.png)

In [6]:
# ✍ Задание 3
def segment_signal(signal, length=4000, overlap=0.75):
    step = int(length * (1 - overlap))
    windows = [signal[i:i+length] for i in range(0, len(signal) - length + 1, step)]
    return np.stack(windows)

segments_normal = segment_signal(normal)
segments_roller = segment_signal(roller)
segments_inner = segment_signal(inner)
segments_outer = segment_signal(outer)
print("Окон:", getattr(segments_normal, "shape", "заполните код"))

Окон: (117, 4000)


In [7]:
check(3, (segments_normal, segments_roller, segments_inner, segments_outer))

✅ Задание 3 выполнено


## Задание 4. Synthetic signal mixtures (синтетические смеси сигналов) по одному признаку

Сделайте 37 примеров: один нормальный и по 12 для внутреннего кольца,
внешнего кольца и ролика. Mixing coefficients (коэффициенты смешивания) заданы ниже.
Для каждого примера используйте linear interpolation (линейную интерполяцию):
`normal + mixing_coefficient * (defect - normal)`. Признак — RMS **после вычитания среднего**.
Цель — три числа `[inner, outer, roller]` по учебной шкале 0–100.
Сохраните `X_matrix` формы `(37, 1)` и `y_matrix` формы `(37, 3)`.

![Synthetic signal mixture (синтетическая смесь сигналов) и RMS](visualizations/03_synthetic_signal_mixture_and_RMS.png)

In [8]:
# ✍ Задание 4
mixing_coefficients = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]

def rms(signal):
    return np.sqrt(np.mean((signal - np.mean(signal)) ** 2))

X_values = [rms(normal)]
y_values = [[0.0, 0.0, 0.0]]
for target_col, defect in enumerate([inner, outer, roller]):
    for mixing_coefficient in mixing_coefficients:
        mixture = normal + mixing_coefficient * (defect - normal)
        X_values.append(rms(mixture))
        target = np.zeros(3)
        target[target_col] = mixing_coefficient * 100
        y_values.append(target)

X_matrix = np.array(X_values).reshape(37, 1)
y_matrix = np.array(y_values)
print("X:", getattr(X_matrix, "shape", None), "y:", getattr(y_matrix, "shape", None))

X: (37, 1) y: (37, 3)


In [9]:
check(4, (X_matrix, y_matrix))

✅ Задание 4 выполнено


## Задание 5. Четырнадцать признаков одного окна

Первые 4000 отсчётов нормального сигнала: RMS, доминирующая частота,
энергии 10 частотных полос по 500 Гц, доля пересечений нуля и отношение
75-го к 25-му процентилю абсолютной амплитуды. Используйте `np.fft.rfft`
и исключите последнюю точку Найквиста, чтобы соответствовать исходному материалу.
Сохраните массив в `features`.

In [10]:
# ✍ Задание 5
N, fs = 4000, 10_000
x = normal[:N] - np.mean(normal[:N])
rms_value = np.sqrt(np.mean(x ** 2))
amplitude = np.abs(np.fft.rfft(x))[:-1]
frequency = np.fft.rfftfreq(N, d=1/fs)[:-1]
dominant_frequency = frequency[np.argmax(amplitude)]
band_energy = np.array([np.sum(amplitude[(frequency >= i * 500) & (frequency < (i + 1) * 500)] ** 2) for i in range(10)])
zero_crossing_rate = np.sum(np.diff(np.sign(x)) != 0) / (len(x) - 1)
q25, q75 = np.quantile(np.abs(x), [0.25, 0.75])
features = np.array([rms_value, dominant_frequency, *band_energy, zero_crossing_rate, q75 / q25])
print("Число признаков:", len(features) if features is not Ellipsis else "заполните код")

Число признаков: 14


In [11]:
check(5, features)

✅ Задание 5 выполнено


## Задание 6. Нормализация признаков

Вычислите среднее и стандартное отклонение **каждого столбца** `X_matrix`.
Если отклонение почти ноль, замените его единицей. Сохраните результат
в `X_scaled`. Используйте `ddof=1`, как в исходном Julia-задании;
самопроверка учитывает эту конвенцию.

In [12]:
# ✍ Задание 6
means = np.mean(X_matrix, axis=0)
stds = np.std(X_matrix, axis=0, ddof=1)
stds[stds < 1e-10] = 1.0
X_scaled = (X_matrix - means) / stds
print("Среднее:", np.mean(X_scaled, axis=0) if X_scaled is not Ellipsis else "заполните код")

Среднее: [1.29025919e-16]


In [13]:
check(6, X_scaled)

✅ Задание 6 выполнено


## Задание 7. Создать три случайных леса

Создайте три `RandomForestRegressor`. Для двух колец: 400 деревьев,
максимальная глубина 35. Для ролика: 600 деревьев, глубина 50.
Добавьте `min_samples_split=2`, `min_samples_leaf=1`, а для
воспроизводимости используйте `random_state=42`, `43`, `44`.

Это именно создание моделей: пока `.fit()` не вызван, деревья ещё не обучены.

In [14]:
# ✍ Задание 7
model_inner = RandomForestRegressor(n_estimators=400, max_depth=35, min_samples_split=2, min_samples_leaf=1, random_state=42)
model_outer = RandomForestRegressor(n_estimators=400, max_depth=35, min_samples_split=2, min_samples_leaf=1, random_state=43)
model_roller = RandomForestRegressor(n_estimators=600, max_depth=50, min_samples_split=2, min_samples_leaf=1, random_state=44)
print("Деревьев:", [m.n_estimators for m in (model_inner, model_outer, model_roller)]
      if model_inner is not Ellipsis else "заполните код")

Деревьев: [400, 400, 600]


In [15]:
check(7, (model_inner, model_outer, model_roller))

✅ Задание 7 выполнено


## Итог и границы метода

Вы построили исходные объекты конвейера. В семинаре мы обучили леса
на более богатом наборе из 54 признаков и проверили на отложенных окнах.
37 примеров и один RMS здесь нужны для понимания формы `X` и `y`,
а не для заявления о качестве диагностики реального оборудования.